# 05 · Retrieval evaluation
Step 6: score retrieval on the 39 answerable gold questions for every chunking strategy (fixed / semantic) and retrieval mode (dense / BM25 / hybrid), through the agent's real analyze → decompose → retrieve steps.

**Cost:** 39 query-analysis calls (`gpt-4.1`, ~1K tokens each, cached in `eval/cache/` after the first run) plus query embeddings (cached). A few cents the first time, about zero after.

**Metrics:** hit@k (any evidence in top k), recall@k (share of evidence items found, matters for comparisons), all@k (every evidence item found), MRR (1 / rank of the first relevant chunk). k = 3, 6, 12; the agent passes 12 chunks to the answer step.

Shut down other notebook kernels first if they have the index open.

In [ ]:
import sys, pathlib
print(sys.executable)
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
pd.set_option('display.width', 200)
from finsight.evaluation.retrieval import run_retrieval_eval, summarize, load_gold, analyze_all

## 1. Query analysis check
Did the agent extract the right companies and years? Compare against the evidence documents.

In [ ]:
gold = [g for g in load_gold() if g.get('evidence')]
analyses = analyze_all(gold)
rows = []
for g in gold:
    a = analyses[g['id']]
    ev_docs = sorted({d for e in g['evidence'] for d in e['docs']})
    rows.append({'id': g['id'], 'question': g['question'][:60], 'companies': a['companies'], 'years': a['fiscal_years'],
                 'in_scope': a['in_scope'], 'evidence_docs': ev_docs})
pd.DataFrame(rows)

## 2. Run the grid

In [ ]:
df = run_retrieval_eval()
print(len(df), 'rows saved to eval/reports/')
summarize(df)

## 3a. Original wording vs paraphrased
Paraphrases use everyday words (capex, EPS, profit, headcount) instead of the filings' labels. Compares the 17 paraphrases with their 17 originals only, so both sides cover the same facts.

In [ ]:
pairs = df[(df['variant'] == 'paraphrase') | df['id'].isin(df.loc[df['variant'] == 'paraphrase', 'paraphrase_of'])]
summarize(pairs, by=('variant', 'strategy', 'mode'))

## 3. By question type (hybrid)

In [ ]:
summarize(df[df['mode'] == 'hybrid'], by=('strategy', 'type'))

## 4. Misses
Questions where hybrid retrieval did not find all evidence in the top 12.

In [ ]:
miss = df[(df['mode'] == 'hybrid') & (df['all@12'] < 1)][['id', 'type', 'strategy', 'companies', 'years', 'recall@12', 'first_rank', 'top_chunks']]
miss

In [ ]:
g_by_id = {g['id']: g for g in gold}
for i in miss['id'].unique():
    print(i, '|', g_by_id[i]['question'], '| evidence:', g_by_id[i]['evidence'])

## 5. Hybrid weight sweep
Dense vs BM25 balance in the fusion. Analyses and query embeddings are cached, so this costs nothing. Compare with the BM25-only rows.

In [ ]:
sweep = []
for w in [(0.5, 0.5), (0.3, 0.7), (0.2, 0.8)]:
    d = run_retrieval_eval(modes=('hybrid',), weights=w, tag=f'hybrid_w{w[0]}')
    s = summarize(d).reset_index(); s.insert(2, 'weights (dense,bm25)', str(w)); sweep.append(s)
bm = summarize(df[df['mode'] == 'bm25']).reset_index(); bm.insert(2, 'weights (dense,bm25)', 'bm25 only')
pd.concat(sweep + [bm]).set_index(['strategy', 'weights (dense,bm25)']).sort_index()

## 6. Reranking impact
Hybrid 0.5/0.5, with and without the LLM reranker (`RERANK_MODEL`). With reranking on, retrieval fetches `RERANK_CANDIDATES` (30) chunks and the reranker keeps the best 12.

**Cost:** one rerank call per question and strategy: 56 × 2 = 112 calls, each reading ~30 chunks (~10–15K tokens), about 3 s each. Run once for the impact analysis.

In [ ]:
rr = []
for r in ('none', 'llm'):
    d = run_retrieval_eval(modes=('hybrid',), rerank=r, tag=f'rerank_{r}')
    rr.append(d)
rr = pd.concat(rr)
summarize(rr, by=('strategy', 'rerank'))

In [ ]:
summarize(rr, by=('strategy', 'rerank', 'type'))

In [ ]:
p = rr[rr['strategy'] == 'fixed'].pivot_table(index='id', columns='rerank', values='all@6')
print('fixed: improved by rerank:', list(p[p['llm'] > p['none']].index))
print('fixed: worse with rerank: ', list(p[p['llm'] < p['none']].index))

## Notes / findings
_(fill in after reviewing)_